In [ ]:
# ============================================
# 03_nlp_exploration.ipynb
# Phase 4: NLP Engine
# ============================================

import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

# Load data
from src.parser import parse_whatsapp_chat
from src.preprocessor import preprocess

df_raw = parse_whatsapp_chat("../data/sample/sample_chat.txt")
df     = preprocess(df_raw)

print("✅ Setup complete!")
print(f"   Loaded {len(df)} messages")

In [ ]:
import nltk
import ssl

# Fix SSL certificate issue on Windows
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Now download again
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

print("✅ All NLTK packages downloaded!")

In [ ]:
import nltk
import os

# Check what NLTK data is already on your machine
nltk_path = r"C:\Users\PRIYANSHI\AppData\Roaming\nltk_data"

for folder in os.listdir(nltk_path):
    subfolder = os.path.join(nltk_path, folder)
    files = os.listdir(subfolder)
    print(f"📁 {folder}: {files}")

In [ ]:
import ssl
import nltk

# Fix SSL
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download one by one with status check
print("Downloading stopwords...")
nltk.download('stopwords', quiet=False)

print("\nDownloading wordnet...")
nltk.download('wordnet', quiet=False)

print("\nDownloading averaged_perceptron_tagger...")
nltk.download('averaged_perceptron_tagger', quiet=False)

print("\n✅ Done!")

In [3]:
# SETUP: 
import sys, os, ssl
sys.path.append(os.path.join(os.getcwd(), ".."))

# SSL fix
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

from src.parser import parse_whatsapp_chat
from src.preprocessor import preprocess

df_raw = parse_whatsapp_chat("../data/sample/sample_chat.txt")
df     = preprocess(df_raw)

print("✅ Setup complete!")
print(f"   Loaded {len(df)} messages")

✅ Setup complete!
   Loaded 25 messages


In [4]:
# TEXT PREPROCESSING PIPELINE
import re

def clean_text(text):
    """Remove URLs, special characters, lowercase."""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_text(text):
    """Full NLP pipeline: clean → tokenize → stopwords → lemmatize."""
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    cleaned    = clean_text(text)
    tokens     = word_tokenize(cleaned)
    filtered   = [w for w in tokens 
                  if w not in stop_words and len(w) > 2]
    lemmatized = [lemmatizer.lemmatize(w) for w in filtered]
    
    return lemmatized

# Test on sample
sample_msg = "Good morning everyone!! How are you all doing today?"
result     = preprocess_text(sample_msg)
print("Original  :", sample_msg)
print("Processed :", result)

# Apply to all normal messages
normal_df  = df[df['message_type'] == 'normal'].copy()
normal_df['processed_tokens'] = normal_df['message'].apply(preprocess_text)
normal_df['processed_text']   = normal_df['processed_tokens'].apply(
    lambda x: ' '.join(x)
)

print(f"\n✅ Preprocessed {len(normal_df)} normal messages")
print("\nSample processed messages:")
for idx, row in normal_df.head(3).iterrows():
    print(f"  Original  : {row['message'][:50]}")
    print(f"  Processed : {row['processed_text'][:50]}")
    print()

Original  : Good morning everyone!! How are you all doing today?
Processed : ['good', 'morning', 'everyone', 'today']

✅ Preprocessed 19 normal messages

Sample processed messages:
  Original  : Hey everyone! Good morning 😊
  Processed : hey everyone good morning

  Original  : Morning! How are you?
  Processed : morning

  Original  : I'm doing great! Just finished my assignment
  Processed : great finished assignment



In [ ]:
# SENTIMENT ANALYSIS: 
def get_sentiment(text):
    """Returns polarity and sentiment label."""
    analysis = TextBlob(str(text))
    polarity = analysis.sentiment.polarity
    
    if polarity > 0.1:
        label = 'positive'
    elif polarity < -0.1:
        label = 'negative'
    else:
        label = 'neutral'
    
    return polarity, label

# Apply to all normal messages
normal_df['polarity']        = normal_df['message'].apply(
    lambda x: get_sentiment(x)[0]
)
normal_df['sentiment_label'] = normal_df['message'].apply(
    lambda x: get_sentiment(x)[1]
)

print("SENTIMENT ANALYSIS RESULTS")
print("=" * 45)
print(normal_df[['sender','message',
                 'polarity','sentiment_label']].to_string())

print("\n📊 Sentiment Distribution:")
print(normal_df['sentiment_label'].value_counts())

print("\n📊 Average Sentiment per User:")
user_sentiment = normal_df.groupby('sender')['polarity'].mean().round(3)
print(user_sentiment.sort_values(ascending=False))

SENTIMENT ANALYSIS RESULTS
     sender                                                             message  polarity sentiment_label
1     Alice                                        Hey everyone! Good morning 😊     0.700        positive
2       Bob                                               Morning! How are you?     0.000         neutral
3     Alice                        I'm doing great! Just finished my assignment     1.000        positive
6     Alice                               Did you all see the project deadline?     0.000         neutral
7     Alice  It's been moved to next Friday which gives us more time thankfully     0.250        positive
8       Bob                                            Oh that's great news! 🎉🎉     1.000        positive
9   Charlie                                     https://github.com/project-repo     0.000         neutral
11     Dave                                                          Hey all! 👋     0.000         neutral
12    Alice        

In [6]:
# TF-IDF KEYWORD EXTRACTION:
# Each user's messages = one document
user_docs = normal_df.groupby('sender')['processed_text'].apply(
    lambda x: ' '.join(x)
).reset_index()
user_docs.columns = ['sender', 'document']

print("Documents created per user:")
for _, row in user_docs.iterrows():
    print(f"  {row['sender']}: '{row['document'][:60]}'")

# Apply TF-IDF
tfidf         = TfidfVectorizer(max_features=20)
tfidf_matrix  = tfidf.fit_transform(user_docs['document'])
feature_names = tfidf.get_feature_names_out()

print("\n🔑 TF-IDF TOP KEYWORDS PER USER:")
print("=" * 45)

for i, sender in enumerate(user_docs['sender']):
    scores       = tfidf_matrix[i].toarray()[0]
    top_indices  = scores.argsort()[::-1][:5]
    top_keywords = [(feature_names[j], round(scores[j], 3))
                    for j in top_indices if scores[j] > 0]
    print(f"\n  {sender}: {top_keywords}")

print("\n✅ NLP Pipeline complete!")

Documents created per user:
  Alice: 'hey everyone good morning great finished assignment see proj'
  Bob: 'morning thats great news good morning everyone got work'
  Charlie: ' ill'
  Dave: 'hey working late tonight look great '

🔑 TF-IDF TOP KEYWORDS PER USER:

  Alice: [('project', 0.476), ('everyone', 0.376), ('morning', 0.376), ('moved', 0.238), ('see', 0.238)]

  Bob: [('morning', 0.59), ('work', 0.374), ('thats', 0.374), ('news', 0.374), ('everyone', 0.295)]

  Charlie: [('ill', 1.0)]

  Dave: [('tonight', 0.498), ('late', 0.498), ('look', 0.498), ('hey', 0.393), ('great', 0.318)]

✅ NLP Pipeline complete!


In [7]:
# Cell 5: Test src/nlp_engine.py
import sys, os
sys.path.append(os.path.join(os.getcwd(), ".."))

from src.nlp_engine import (
    run_nlp_pipeline,
    get_sentiment_stats,
    get_tfidf_keywords
)

# Run pipeline
nlp_df   = run_nlp_pipeline(df)
sentiment = get_sentiment_stats(nlp_df)
keywords  = get_tfidf_keywords(nlp_df)

print("✅ run_nlp_pipeline:")
print(f"   Processed {len(nlp_df)} messages")

print("\n✅ get_sentiment_stats:")
print(f"   Overall polarity  : {sentiment['overall_polarity']}")
print(f"   Most positive user: {sentiment['user_sentiment'].index[0]}")

print("\n✅ get_tfidf_keywords:")
for user, kws in keywords.items():
    print(f"   {user}: {kws[:2]}")

print("\n🎉 nlp_engine.py is working perfectly!")

✅ run_nlp_pipeline:
   Processed 19 messages

✅ get_sentiment_stats:
   Overall polarity  : 0.238
   Most positive user: Bob

✅ get_tfidf_keywords:
   Alice: [('project', 0.476), ('everyone', 0.376)]
   Bob: [('morning', 0.59), ('work', 0.374)]
   Charlie: [('ill', 1.0)]
   Dave: [('tonight', 0.498), ('late', 0.498)]

🎉 nlp_engine.py is working perfectly!
